In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer
from recommender import *

In [2]:
# Path variables
TRIPLES_PATH = Path('../data/triples_new_without_ct_ss.csv')
RECIPE_PATH_CSV = Path("../data/dataFullLargerRegionAndCountryWithServingsBin.csv")

In [4]:
def tuple_to_canonical(s: str) -> str:
    """Convert ('recipe', '123') → 'recipe_id_123', else 'rel_val'."""
    try:
        head, tail = ast.literal_eval(s)
        tail = str(tail).strip()
        return f"{head}_id_{tail}" if head == "recipe" else f"{head}_{tail}"
    except Exception:
        return s.strip()

In [5]:
def build_recipe_texts(triples_csv: Path) -> pd.DataFrame:
    """Aggregate recipe triples into full_text per recipe_id."""
    df = pd.read_csv(triples_csv, dtype=str)
    df["Head"] = df["Head"].apply(tuple_to_canonical)
    df["Tail"] = df["Tail"].apply(tuple_to_canonical)
    df["Relation"] = df["Relation"].str.strip()

    recipe_triples = df[df["Head"].str.startswith("recipe_")]
    grouped = (
        recipe_triples.groupby("Head")
        .apply(
            lambda g: ". ".join(f"{r} {t}" for r, t in zip(g["Relation"], g["Tail"]))
        )
        .reset_index(name="full_text")
        .rename(columns={"Head": "recipe_id"})
        .set_index("recipe_id")
    )
    return grouped

In [6]:
def embed_recipes(texts: pd.Series, model_name: str, batch_size: int) -> np.ndarray:
    """Encode recipe texts to embeddings."""
    model = SentenceTransformer(model_name)
    embs = model.encode(texts.tolist(), batch_size=batch_size, show_progress_bar=True)
    return embs, model

In [ ]:
def get_recipe_embeddings(TRIPLES_PATH):
    # Build & embed recipes
    rec_texts = build_recipe_texts(TRIPLES_PATH)
    recipe_ids = rec_texts.index.tolist()
    recipe_embs, model = embed_recipes(rec_texts["full_text"], "all-MiniLM-L6-v2", 64)

    # Save recipe ids
    np.save(f"../results/recipe_ids.npy", recipe_ids)

    # Save embeddings
    np.save(f"../results/MiniLM_recipe_embs.npy", recipe_embs)

    # Save model
    model.save(f"../results/MiniLM_model")